In [15]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import joblib
import threading
import traceback

class ClassicFuelApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Прогноз расхода топлива по контракту")
        self.root.geometry("1350x900")
        
        # Файлы
        self.buses_file = "Районы_автобусы.csv"
        self.model_file = "best_model.pkl"
        self.le_bus_file = "le_bus.pkl"
        self.le_rayon_file = "le_rayon.pkl"
        self.fuel_norms_file = "fuel_norms_all.csv"
        
        # Данные
        self.df_routes = pd.DataFrame(columns=[
            'contract_number', 'route_number', 'direction', 'route_name',
            'distance_km', 'bus_class', 'buses_count', 'bus_plates', 'rayon'
        ])
        self.df_schedule = pd.DataFrame(columns=[
            'contract_number', 'route_number', 'direction',
            'days_pattern', 'custom_days', 'departure_time'
        ])
        
        # Модель
        self.model = None
        self.le_bus = None
        self.le_rayon = None
        self.feature_names = None
        
        # Словарь норм расхода
        self.bus_fuel_norms = {}
        
        self.status_var = tk.StringVar(value="Готов")
        self.current_selected_route = None
        
        self.load_buses()
        self.load_fuel_norms()
        self.load_model()
        self.create_widgets()
        self.refresh_tables()
    
    def load_buses(self):
        if os.path.exists(self.buses_file):
            self.df_buses = pd.read_csv(self.buses_file, sep=';')
            self.buses_by_rayon = {}
            for rayon in self.df_buses['Район'].unique():
                self.buses_by_rayon[rayon] = self.df_buses[self.df_buses['Район'] == rayon]['Гос. Номер'].tolist()
            print(f"✅ Загружено {len(self.df_buses)} автобусов")
        else:
            self.buses_by_rayon = {}
            print("⚠️ Файл Районы_автобусы.csv не найден")
    
    def load_fuel_norms(self):
        if os.path.exists(self.fuel_norms_file):
            try:
                df_norms = pd.read_csv(self.fuel_norms_file, encoding='utf-8-sig')
                print(f"✅ Загружен {self.fuel_norms_file}: {len(df_norms)} записей")
                
                df_norms['Дата'] = pd.to_datetime(df_norms['Дата'], format='%d.%m.%Y')
                df_latest = df_norms.sort_values('Дата').groupby('Гос_номер').last().reset_index()
                self.bus_fuel_norms = dict(zip(df_latest['Гос_номер'], df_latest['Норма_расхода_л']))
                
                print(f"✅ Загружены нормы для {len(self.bus_fuel_norms)} автобусов")
                sample = list(self.bus_fuel_norms.items())[:5]
                for bus, norm in sample:
                    print(f"   {bus}: {norm} л/100км")
            except Exception as e:
                print(f"❌ Ошибка загрузки норм: {e}")
                self.bus_fuel_norms = {}
        else:
            print(f"⚠️ Файл {self.fuel_norms_file} не найден")
            self.bus_fuel_norms = {}
    
    def get_fuel_norm(self, bus_plate, bus_class):
        if bus_plate in self.bus_fuel_norms:
            return float(self.bus_fuel_norms[bus_plate])
        
        default_norms = {
            'Малый': 22.0,
            'Средний': 28.0,
            'Большой': 35.0
        }
        return default_norms.get(bus_class, 25.0)
    
    def load_model(self):
        if os.path.exists(self.model_file):
            self.model = joblib.load(self.model_file)
            print(f"✅ Модель загружена: {self.model_file}")
            if hasattr(self.model, 'feature_names_in_'):
                self.feature_names = list(self.model.feature_names_in_)
                print(f"   Модель ожидает признаки: {self.feature_names}")
        else:
            print(f"⚠️ Файл модели {self.model_file} не найден")
            
        if os.path.exists(self.le_bus_file):
            self.le_bus = joblib.load(self.le_bus_file)
            print(f"✅ Энкодер автобусов загружен")
        else:
            print(f"⚠️ Файл {self.le_bus_file} не найден")
            
        if os.path.exists(self.le_rayon_file):
            self.le_rayon = joblib.load(self.le_rayon_file)
            print(f"✅ Энкодер районов загружен")
        else:
            print(f"⚠️ Файл {self.le_rayon_file} не найден")
    
    def create_widgets(self):
        top = ttk.Frame(self.root)
        top.pack(fill=tk.X, padx=10, pady=5)
        ttk.Button(top, text="💾 Сохранить данные", command=self.save_data).pack(side=tk.LEFT, padx=5)
        ttk.Button(top, text="📂 Загрузить данные", command=self.load_data).pack(side=tk.LEFT, padx=5)
        ttk.Button(top, text="🚀 РАССЧИТАТЬ ПРОГНОЗ", command=self.run_prediction, width=18).pack(side=tk.LEFT, padx=10)
        ttk.Label(top, textvariable=self.status_var, foreground='blue').pack(side=tk.RIGHT)
        
        main = ttk.Frame(self.root)
        main.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        
        # ===== ЛЕВАЯ КОЛОНКА =====
        left = ttk.LabelFrame(main, text="📋 Маршруты (информация)", padding=5)
        left.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        columns = ('contract', 'route', 'dir', 'name', 'km', 'class', 'buses', 'plates')
        self.tree = ttk.Treeview(left, columns=columns, show='headings', height=10)
        self.tree.heading('contract', text='Контракт')
        self.tree.heading('route', text='№')
        self.tree.heading('dir', text='Напр')
        self.tree.heading('name', text='Название')
        self.tree.heading('km', text='км')
        self.tree.heading('class', text='Класс')
        self.tree.heading('buses', text='Кол-во')
        self.tree.heading('plates', text='Госномера')
        for col in columns:
            self.tree.column(col, width=100)
        self.tree.pack(side=tk.TOP, fill=tk.BOTH, expand=True)
        self.tree.bind('<<TreeviewSelect>>', self.on_tree_select)
        
        form = ttk.LabelFrame(left, text="Добавить/Редактировать маршрут", padding=5)
        form.pack(side=tk.BOTTOM, fill=tk.X, pady=5)
        
        ttk.Label(form, text="Контракт:").grid(row=0, column=0, sticky=tk.W)
        self.contract_entry = ttk.Entry(form, width=20)
        self.contract_entry.grid(row=0, column=1, padx=5)
        
        ttk.Label(form, text="№ маршрута:").grid(row=0, column=2, sticky=tk.W)
        self.route_entry = ttk.Entry(form, width=8)
        self.route_entry.grid(row=0, column=3, padx=5)
        
        ttk.Label(form, text="Направление:").grid(row=0, column=4, sticky=tk.W)
        self.dir_var = tk.StringVar(value="прямое")
        ttk.Combobox(form, textvariable=self.dir_var, values=['прямое','обратное'], width=8).grid(row=0, column=5, padx=5)
        
        ttk.Label(form, text="Название:").grid(row=1, column=0, sticky=tk.W)
        self.name_entry = ttk.Entry(form, width=30)
        self.name_entry.grid(row=1, column=1, columnspan=3, padx=5, sticky=tk.W)
        
        ttk.Label(form, text="Расстояние (км):").grid(row=1, column=4, sticky=tk.W)
        self.dist_entry = ttk.Entry(form, width=8)
        self.dist_entry.grid(row=1, column=5, padx=5)
        
        ttk.Label(form, text="Класс автобуса:").grid(row=2, column=0, sticky=tk.W)
        self.class_combo = ttk.Combobox(form, values=['Малый','Средний','Большой'], width=12)
        self.class_combo.grid(row=2, column=1, padx=5)
        
        ttk.Label(form, text="Кол-во автобусов:").grid(row=2, column=2, sticky=tk.W)
        self.buses_spin = ttk.Spinbox(form, from_=1, to=10, width=5, command=self.on_buses_count_change)
        self.buses_spin.bind('<KeyRelease>', self.on_buses_count_change)
        self.buses_spin.grid(row=2, column=3, padx=5)
        
        ttk.Label(form, text="Район:").grid(row=2, column=4, sticky=tk.W)
        self.rayon_combo = ttk.Combobox(form, values=list(self.buses_by_rayon.keys()), width=15)
        self.rayon_combo.grid(row=2, column=5, padx=5)
        self.rayon_combo.bind('<<ComboboxSelected>>', self.update_bus_list)
        
        ttk.Label(form, text="Автобусы:").grid(row=3, column=0, sticky=tk.NW)
        bus_frame = ttk.Frame(form)
        bus_frame.grid(row=3, column=1, columnspan=3, sticky=tk.W)
        
        self.bus_listbox = tk.Listbox(bus_frame, selectmode=tk.MULTIPLE, height=4, width=35)
        self.bus_listbox.pack(side=tk.LEFT)
        self.bus_listbox.bind('<<ListboxSelect>>', lambda e: self.on_buses_count_change())
        
        bus_scroll = ttk.Scrollbar(bus_frame, orient=tk.VERTICAL, command=self.bus_listbox.yview)
        self.bus_listbox.configure(yscrollcommand=bus_scroll.set)
        bus_scroll.pack(side=tk.RIGHT, fill=tk.Y)
        
        self.selected_buses_label = ttk.Label(form, text="✅ Выбрано: 0", foreground='green')
        self.selected_buses_label.grid(row=3, column=4, columnspan=2, sticky=tk.W, padx=5)
        
        btn_frame = ttk.Frame(form)
        btn_frame.grid(row=4, column=0, columnspan=6, pady=10)
        ttk.Button(btn_frame, text="➕ Добавить маршрут", command=self.add_route).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="🗑️ Удалить выбранный", command=self.delete_route).pack(side=tk.LEFT, padx=5)
        
        # ===== СРЕДНЯЯ КОЛОНКА =====
        middle = ttk.LabelFrame(main, text="📅 Период выполнения перевозок", padding=5)
        middle.pack(side=tk.LEFT, fill=tk.Y, padx=5)
        
        ttk.Label(middle, text="Дата начала:").pack(anchor=tk.W, pady=(5,0))
        self.start_date_entry = ttk.Entry(middle, width=12)
        self.start_date_entry.pack(anchor=tk.W, pady=(0,5))
        self.start_date_entry.insert(0, "01.01.2025")
        
        ttk.Label(middle, text="Дата окончания:").pack(anchor=tk.W, pady=(5,0))
        self.end_date_entry = ttk.Entry(middle, width=12)
        self.end_date_entry.pack(anchor=tk.W, pady=(0,5))
        self.end_date_entry.insert(0, "31.12.2025")
        
        ttk.Label(middle, text="Пример: 01.01.2025", font=('Arial', 8)).pack(anchor=tk.W, pady=5)
        
        # ===== ПРАВАЯ КОЛОНКА =====
        right = ttk.LabelFrame(main, text="⏰ Расписание", padding=5)
        right.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)
        
        ttk.Label(right, text="Выберите маршрут:").pack(anchor=tk.W, pady=(0,5))
        self.route_combo = ttk.Combobox(right, width=50)
        self.route_combo.pack(fill=tk.X, pady=(0,10))
        self.route_combo.bind('<<ComboboxSelected>>', self.on_route_select)
        
        ttk.Button(right, text="🔄 Обновить список", command=self.update_route_combo).pack(pady=(0,10))
        
        days_frame = ttk.LabelFrame(right, text="Дни выполнения", padding=5)
        days_frame.pack(fill=tk.X, pady=5)
        
        self.days_pattern = tk.StringVar(value="everyday")
        patterns = [("Ежедневно", "everyday"), ("Только будни", "weekdays"), 
                    ("Только выходные", "weekend"), ("Пн, Ср, Пт", "mon_wed_fri"),
                    ("Вт, Чт", "tue_thu"), ("Свои дни", "custom")]
        
        row, col = 0, 0
        for text, val in patterns:
            ttk.Radiobutton(days_frame, text=text, variable=self.days_pattern, 
                           value=val, command=self.on_days_change).grid(row=row, column=col, padx=5, sticky=tk.W)
            col += 1
            if col > 2:
                col = 0
                row += 1
        
        # Поле для своих дней - всегда видимо
        self.custom_frame = ttk.Frame(days_frame)
        self.custom_frame.grid(row=row+1, column=0, columnspan=3, pady=5, sticky=tk.W)
        ttk.Label(self.custom_frame, text="Дни (0=Пн..6=Вс):").pack(side=tk.LEFT)
        self.custom_entry = ttk.Entry(self.custom_frame, width=20)
        self.custom_entry.pack(side=tk.LEFT, padx=5)
        ttk.Label(self.custom_frame, text="Пример: 0,5 (пн и сб)").pack(side=tk.LEFT)
        
        time_frame = ttk.LabelFrame(right, text="Время отправления", padding=5)
        time_frame.pack(fill=tk.X, pady=5)
        
        ttk.Label(time_frame, text="ЧЧ:ММ:").pack(side=tk.LEFT, padx=5)
        self.time_entry = ttk.Entry(time_frame, width=10)
        self.time_entry.pack(side=tk.LEFT, padx=5)
        ttk.Button(time_frame, text="➕ Добавить время", command=self.add_schedule).pack(side=tk.LEFT, padx=10)
        
        self.current_route_label = ttk.Label(right, text="", font=('Arial', 9, 'bold'), foreground='blue')
        self.current_route_label.pack(pady=5)
        
        self.schedule_tree = ttk.Treeview(right, columns=('time', 'days'), show='headings', height=10)
        self.schedule_tree.heading('time', text='Время')
        self.schedule_tree.heading('days', text='Дни выполнения')
        self.schedule_tree.column('time', width=80)
        self.schedule_tree.column('days', width=200)
        self.schedule_tree.pack(fill=tk.BOTH, expand=True, pady=5)
        
        ttk.Button(right, text="🗑️ Удалить выбранное время", command=self.delete_schedule).pack(pady=5)
        
        results_frame = ttk.LabelFrame(self.root, text="📊 Результаты прогноза", padding=5)
        results_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=5)
        
        self.results_text = tk.Text(results_frame, wrap=tk.WORD, height=8, font=('Courier', 9))
        self.results_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        scroll = ttk.Scrollbar(results_frame, orient=tk.VERTICAL, command=self.results_text.yview)
        self.results_text.configure(yscrollcommand=scroll.set)
        scroll.pack(side=tk.RIGHT, fill=tk.Y)
    
    def on_buses_count_change(self, event=None):
        try:
            needed = int(self.buses_spin.get())
        except:
            needed = 1
        selected = len(self.get_selected_buses())
        self.selected_buses_label.config(text=f"✅ Выбрано: {selected} из {needed}")
        if selected < needed:
            self.selected_buses_label.config(foreground='red')
        else:
            self.selected_buses_label.config(foreground='green')
    
    def update_bus_list(self, event):
        rayon = self.rayon_combo.get()
        if rayon in self.buses_by_rayon:
            self.bus_listbox.delete(0, tk.END)
            for bus in self.buses_by_rayon[rayon]:
                self.bus_listbox.insert(tk.END, bus)
            self.bus_listbox.selection_clear(0, tk.END)
            self.on_buses_count_change()
    
    def get_selected_buses(self):
        return [self.bus_listbox.get(i) for i in self.bus_listbox.curselection()]
    
    def on_days_change(self):
        # Ничего не делаем, поле всегда видимо
        pass
    
    def update_route_combo(self):
        routes = [f"{row['contract_number']} | {row['route_number']} | {row['direction']} | {row['route_name']}" 
                  for _, row in self.df_routes.iterrows()]
        self.route_combo['values'] = routes
    
    def on_tree_select(self, event):
        selected = self.tree.selection()
        if not selected:
            return
        
        idx = self.tree.index(selected[0])
        row = self.df_routes.iloc[idx]
        
        self.contract_entry.delete(0, tk.END)
        self.contract_entry.insert(0, row['contract_number'])
        self.route_entry.delete(0, tk.END)
        self.route_entry.insert(0, row['route_number'])
        self.dir_var.set(row['direction'])
        self.name_entry.delete(0, tk.END)
        self.name_entry.insert(0, row['route_name'])
        self.dist_entry.delete(0, tk.END)
        self.dist_entry.insert(0, row['distance_km'])
        self.class_combo.set(row['bus_class'])
        self.buses_spin.delete(0, tk.END)
        self.buses_spin.insert(0, row['buses_count'])
        
        if 'rayon' in row and row.get('rayon'):
            self.rayon_combo.set(row['rayon'])
            self.update_bus_list(None)
        
        if 'bus_plates' in row and row.get('bus_plates'):
            plates = row['bus_plates'].split(',')
            self.bus_listbox.selection_clear(0, tk.END)
            for i, bus in enumerate(self.bus_listbox.get(0, tk.END)):
                if bus in plates:
                    self.bus_listbox.selection_set(i)
        
        self.on_buses_count_change()
    
    def on_route_select(self, event):
        selected = self.route_combo.get()
        if selected:
            self.current_selected_route = selected
            self.current_route_label.config(text=f"📌 Выбран: {selected}")
            self.refresh_schedule_table()
    
    def refresh_tables(self):
        for item in self.tree.get_children():
            self.tree.delete(item)
        for _, row in self.df_routes.iterrows():
            plates = row.get('bus_plates', '')[:20]
            self.tree.insert('', tk.END, values=(
                row['contract_number'][:12], row['route_number'], row['direction'],
                row['route_name'][:20], row['distance_km'], row['bus_class'],
                row['buses_count'], plates
            ))
        self.update_route_combo()
    
    def refresh_schedule_table(self):
        for item in self.schedule_tree.get_children():
            self.schedule_tree.delete(item)
        
        if not self.current_selected_route:
            return
        
        parts = self.current_selected_route.split(' | ')
        if len(parts) < 3:
            return
        
        contract, route_num, direction = parts[0], parts[1], parts[2]
        
        mask = (self.df_schedule['contract_number'] == contract) & \
               (self.df_schedule['route_number'] == route_num) & \
               (self.df_schedule['direction'] == direction)
        
        for _, row in self.df_schedule[mask].iterrows():
            if row['days_pattern'] == 'custom' and row.get('custom_days'):
                desc = f"Свои дни: {row['custom_days']} (0=Пн..6=Вс)"
            else:
                desc = {'everyday':'Ежедневно', 'weekdays':'Пн-Пт (будни)', 'weekend':'Сб-Вс (выходные)',
                        'mon_wed_fri':'Пн, Ср, Пт', 'tue_thu':'Вт, Чт'}.get(row['days_pattern'], f"Дни: {row.get('custom_days','')}")
            self.schedule_tree.insert('', tk.END, values=(row['departure_time'], desc))
    
    def add_route(self):
        contract = self.contract_entry.get().strip()
        route_num = self.route_entry.get().strip()
        direction = self.dir_var.get()
        name = self.name_entry.get().strip()
        distance = self.dist_entry.get().strip()
        rayon = self.rayon_combo.get()
        selected_buses = self.get_selected_buses()
        
        try:
            buses_count = int(self.buses_spin.get())
        except:
            buses_count = 1
        
        if not all([contract, route_num, name, distance]):
            messagebox.showwarning("Ошибка", "Заполните: контракт, номер, название, расстояние!")
            return
        
        if len(selected_buses) < buses_count:
            messagebox.showwarning("Ошибка", f"Нужно выбрать {buses_count} автобусов (выбрано {len(selected_buses)})!")
            return
        
        try:
            dist = float(distance.replace(',', '.'))
        except:
            messagebox.showwarning("Ошибка", "Расстояние должно быть числом!")
            return
        
        mask = (self.df_routes['contract_number'] == contract) & \
               (self.df_routes['route_number'] == route_num) & \
               (self.df_routes['direction'] == direction)
        
        new_row = {
            'contract_number': contract, 'route_number': route_num, 'direction': direction,
            'route_name': name, 'distance_km': dist, 'bus_class': self.class_combo.get(),
            'buses_count': buses_count, 'bus_plates': ','.join(selected_buses), 'rayon': rayon
        }
        
        if mask.any():
            idx = self.df_routes[mask].index[0]
            self.df_routes.loc[idx] = new_row
            messagebox.showinfo("Успех", f"Маршрут {route_num} ({direction}) обновлен!")
        else:
            self.df_routes = pd.concat([self.df_routes, pd.DataFrame([new_row])], ignore_index=True)
            messagebox.showinfo("Успех", f"Маршрут {route_num} ({direction}) добавлен!")
        
        self.refresh_tables()
    
    def delete_route(self):
        selected = self.tree.selection()
        if not selected:
            messagebox.showwarning("Ошибка", "Выберите маршрут в таблице!")
            return
        
        idx = self.tree.index(selected[0])
        row = self.df_routes.iloc[idx]
        
        if messagebox.askyesno("Удаление", f"Удалить маршрут {row['route_number']} ({row['direction']})?"):
            self.df_routes = self.df_routes.drop(idx).reset_index(drop=True)
            mask = (self.df_schedule['contract_number'] == row['contract_number']) & \
                   (self.df_schedule['route_number'] == row['route_number']) & \
                   (self.df_schedule['direction'] == row['direction'])
            self.df_schedule = self.df_schedule[~mask].reset_index(drop=True)
            self.refresh_tables()
            self.refresh_schedule_table()
    
    def add_schedule(self):
        if not self.current_selected_route:
            messagebox.showwarning("Ошибка", "Сначала выберите маршрут в правой панели!")
            return
        
        time_str = self.time_entry.get().strip()
        if not time_str:
            messagebox.showwarning("Ошибка", "Введите время!")
            return
        
        try:
            datetime.strptime(time_str, '%H:%M')
        except:
            messagebox.showwarning("Ошибка", "Неверный формат! Используйте ЧЧ:ММ (08:30)")
            return
        
        parts = self.current_selected_route.split(' | ')
        contract, route_num, direction = parts[0], parts[1], parts[2]
        days_pattern = self.days_pattern.get()
        
        # Всегда берем значение из поля custom_entry
        custom_days = self.custom_entry.get().strip() if days_pattern == 'custom' else ''
        
        mask = (self.df_schedule['contract_number'] == contract) & \
               (self.df_schedule['route_number'] == route_num) & \
               (self.df_schedule['direction'] == direction) & \
               (self.df_schedule['departure_time'] == time_str)
        
        if mask.any():
            messagebox.showwarning("Ошибка", "Это время уже добавлено!")
            return
        
        new_row = {
            'contract_number': contract, 'route_number': route_num, 'direction': direction,
            'days_pattern': days_pattern, 'custom_days': custom_days, 'departure_time': time_str
        }
        
        self.df_schedule = pd.concat([self.df_schedule, pd.DataFrame([new_row])], ignore_index=True)
        self.refresh_schedule_table()
        self.time_entry.delete(0, tk.END)
        messagebox.showinfo("Успех", f"Добавлено время {time_str}")
    
    def delete_schedule(self):
        selected = self.schedule_tree.selection()
        if not selected:
            messagebox.showwarning("Ошибка", "Выберите время в расписании!")
            return
        
        time_str = self.schedule_tree.item(selected[0])['values'][0]
        parts = self.current_selected_route.split(' | ')
        contract, route_num, direction = parts[0], parts[1], parts[2]
        
        mask = (self.df_schedule['contract_number'] == contract) & \
               (self.df_schedule['route_number'] == route_num) & \
               (self.df_schedule['direction'] == direction) & \
               (self.df_schedule['departure_time'] == time_str)
        
        self.df_schedule = self.df_schedule[~mask].reset_index(drop=True)
        self.refresh_schedule_table()
        messagebox.showinfo("Успех", f"Время {time_str} удалено!")
    
    def save_data(self):
        if self.df_routes.empty:
            messagebox.showwarning("Ошибка", "Нет данных для сохранения!")
            return
        filepath = filedialog.asksaveasfilename(defaultextension=".xlsx", filetypes=[("Excel", "*.xlsx")])
        if filepath:
            with pd.ExcelWriter(filepath) as writer:
                self.df_routes.to_excel(writer, sheet_name='Маршруты', index=False)
                self.df_schedule.to_excel(writer, sheet_name='Расписание', index=False)
            messagebox.showinfo("Сохранено", f"Сохранено в {filepath}")
    
    def load_data(self):
        filepath = filedialog.askopenfilename(filetypes=[("Excel", "*.xlsx")])
        if filepath:
            self.df_routes = pd.read_excel(filepath, sheet_name='Маршруты')
            self.df_schedule = pd.read_excel(filepath, sheet_name='Расписание')
            
            # Приводим к строке (вот это лечит ошибку)
            self.df_routes['contract_number'] = self.df_routes['contract_number'].astype(str)
            self.df_routes['route_number'] = self.df_routes['route_number'].astype(str)
            self.df_schedule['contract_number'] = self.df_schedule['contract_number'].astype(str)
            self.df_schedule['route_number'] = self.df_schedule['route_number'].astype(str)
            
            self.refresh_tables()
            self.refresh_schedule_table()
            messagebox.showinfo("Загружено", f"Загружено {len(self.df_routes)} маршрутов, {len(self.df_schedule)} расписаний")
    
    def _update_results_safe(self, text):
        self.root.after(0, lambda: self._update_results(text))
    
    def _update_results(self, text):
        self.results_text.insert(tk.END, text)
        self.results_text.see(tk.END)
        self.root.update_idletasks()
    
    def run_prediction(self):
        if self.df_routes.empty:
            messagebox.showwarning("Ошибка", "Нет данных о маршрутах!")
            return
        if self.model is None:
            messagebox.showwarning("Ошибка", "Модель не загружена!")
            return
        
        try:
            start_date = datetime.strptime(self.start_date_entry.get().strip(), '%d.%m.%Y')
            end_date = datetime.strptime(self.end_date_entry.get().strip(), '%d.%m.%Y')
            if start_date > end_date:
                messagebox.showwarning("Ошибка", "Дата начала позже даты окончания!")
                return
        except:
            messagebox.showwarning("Ошибка", "Неверный формат даты! Используйте ДД.ММ.ГГГГ (например, 01.01.2025)")
            return
        
        self.status_var.set("Расчет...")
        self.results_text.delete(1.0, tk.END)
        
        thread = threading.Thread(target=self._predict, args=(start_date, end_date), daemon=True)
        thread.start()
    
    def _predict(self, start_date, end_date):
        try:
            contract = self.df_routes['contract_number'].iloc[0]
            self._update_results_safe(f"🚀 Генерация поездок для контракта {contract}\n")
            self._update_results_safe(f"📅 Период: {start_date.strftime('%d.%m.%Y')} — {end_date.strftime('%d.%m.%Y')}\n\n")
            
            trips = []
            norms_log = {}
            bus_index_counter = {}
            
            def is_day_matching(date, days_pattern, custom_days):
                weekday = date.weekday()
                if days_pattern == 'everyday':
                    return True
                elif days_pattern == 'weekdays':
                    return weekday < 5
                elif days_pattern == 'weekend':
                    return weekday >= 5
                elif days_pattern == 'mon_wed_fri':
                    return weekday in [0, 2, 4]
                elif days_pattern == 'tue_thu':
                    return weekday in [1, 3]
                elif days_pattern == 'custom' and custom_days:
                    try:
                        allowed = [int(d.strip()) for d in str(custom_days).split(',') if d.strip().isdigit()]
                        return weekday in allowed
                    except:
                        return False
                return False
            
            schedule_by_route = {}
            for _, sched in self.df_schedule.iterrows():
                key = (sched['contract_number'], sched['route_number'], sched['direction'])
                if key not in schedule_by_route:
                    schedule_by_route[key] = []
                schedule_by_route[key].append(sched)
            
            current_date = start_date
            total_days = 0
            
            self._update_results_safe("📋 Используемые нормы расхода:\n")
            
            while current_date <= end_date:
                total_days += 1
                if total_days % 30 == 0:
                    self._update_results_safe(f"   Обработано {total_days} дней, сгенерировано {len(trips)} поездок...\n")
                
                for _, route in self.df_routes.iterrows():
                    route_num = route['route_number']
                    direction = route['direction']
                    distance = route['distance_km']
                    bus_class = route['bus_class']
                    bus_plates = [b.strip() for b in str(route.get('bus_plates', '')).split(',') if b.strip() and b.strip() != 'nan']
                    rayon = route.get('rayon', '')
                    contract_num = route['contract_number']
                    
                    if not bus_plates:
                        continue
                    
                    key = (contract_num, route_num, direction)
                    schedules = schedule_by_route.get(key, [])
                    
                    if not schedules:
                        continue
                    
                    counter_key = f"{contract_num}_{route_num}_{direction}"
                    sorted_schedules = sorted(schedules, key=lambda x: x['departure_time'])
                    
                    for sched in sorted_schedules:
                        days_pattern = sched['days_pattern']
                        custom_days = sched.get('custom_days', '')
                        departure_time = sched['departure_time']
                        
                        if is_day_matching(current_date, days_pattern, custom_days):
                            try:
                                if ':' in str(departure_time):
                                    hour, minute = map(int, str(departure_time).split(':'))
                                else:
                                    hour, minute = int(departure_time), 0
                                
                                trip_time = datetime(current_date.year, current_date.month, 
                                                   current_date.day, hour, minute)
                                
                                if counter_key not in bus_index_counter:
                                    bus_index_counter[counter_key] = 0
                                
                                bus_idx = bus_index_counter[counter_key] % len(bus_plates)
                                selected_bus = bus_plates[bus_idx]
                                bus_index_counter[counter_key] += 1
                                
                                norm_fuel = self.get_fuel_norm(selected_bus, bus_class)
                                
                                if selected_bus not in norms_log:
                                    norms_log[selected_bus] = norm_fuel
                                
                                trips.append({
                                    'Расстояние_км': distance,
                                    'Госномер': selected_bus,
                                    'Район': rayon,
                                    'Класс_автобуса': bus_class,
                                    'Час': trip_time.hour,
                                    'День_недели': trip_time.weekday(),
                                    'Выходной': 1 if trip_time.weekday() >= 5 else 0,
                                    'Месяц': trip_time.month,
                                    'is_winter': 1 if trip_time.month in [12, 1, 2] else 0,
                                    'is_summer': 1 if trip_time.month in [6, 7, 8] else 0,
                                    'is_night': 1 if trip_time.hour in [23, 0, 1, 2, 3, 4, 5] else 0,
                                    'is_rush_hour': 1 if ((trip_time.hour >= 7 and trip_time.hour <= 10) or 
                                                           (trip_time.hour >= 17 and trip_time.hour <= 20)) else 0,
                                    'Норма_расхода_л': norm_fuel,
                                    'Дата': trip_time.strftime('%Y-%m-%d')
                                })
                            except Exception as e:
                                pass
                
                current_date += timedelta(days=1)
            
            self._update_results_safe(f"\n📊 Нормы расхода (из файла, по самой поздней дате):\n")
            for bus, norm in list(norms_log.items())[:10]:
                self._update_results_safe(f"   {bus}: {norm} л/100км\n")
            if len(norms_log) > 10:
                self._update_results_safe(f"   ... и еще {len(norms_log) - 10} автобусов\n")
            
            df = pd.DataFrame(trips)
            
            if df.empty:
                self._update_results_safe("❌ Нет поездок за указанный период! Проверьте расписание.\n")
                self.root.after(0, lambda: self.status_var.set("Ошибка: нет поездок"))
                return
            
            df['Месяц_синус'] = np.sin(2 * np.pi * df['Месяц'] / 12)
            df['Месяц_косинус'] = np.cos(2 * np.pi * df['Месяц'] / 12)
            
            if self.le_bus:
                bus_map = {}
                for bus in df['Госномер'].unique():
                    if bus in self.le_bus.classes_:
                        bus_map[bus] = self.le_bus.transform([bus])[0]
                    else:
                        bus_map[bus] = 0
                df['bus_code'] = df['Госномер'].map(bus_map)
            else:
                df['bus_code'] = 0
            
            if self.le_rayon:
                rayon_map = {}
                for r in df['Район'].unique():
                    if r and r in self.le_rayon.classes_:
                        rayon_map[r] = self.le_rayon.transform([r])[0]
                    else:
                        rayon_map[r] = 0
                df['rayon_code'] = df['Район'].map(rayon_map)
            else:
                df['rayon_code'] = 0
            
            if self.feature_names:
                features = self.feature_names
            else:
                features = ['Distance (km)', 'Hour', 'Day of the week', 'Weekend',
                           'Month (sin)', 'Month (cos)', 'Bus code', 'District code',
                           'Consumption rate (l/100km)', 'is winter', 'is summer', 'is night', 'is rush hour']
            
            df_model = pd.DataFrame()
            df_model['Distance (km)'] = df['Расстояние_км']
            df_model['Hour'] = df['Час']
            df_model['Day of the week'] = df['День_недели']
            df_model['Weekend'] = df['Выходной']
            df_model['Month (sin)'] = df['Месяц_синус']
            df_model['Month (cos)'] = df['Месяц_косинус']
            df_model['Bus code'] = df['bus_code']
            df_model['District code'] = df['rayon_code']
            df_model['Consumption rate (l/100km)'] = df['Норма_расхода_л']
            df_model['is winter'] = df['is_winter']
            df_model['is summer'] = df['is_summer']
            df_model['is night'] = df['is_night']
            df_model['is rush hour'] = df['is_rush_hour']
            
            missing = [f for f in features if f not in df_model.columns]
            if missing:
                self._update_results_safe(f"❌ Отсутствуют признаки: {missing}\n")
                self.root.after(0, lambda: self.status_var.set("Ошибка"))
                return
            
            self._update_results_safe(f"\n   Выполняется прогноз для {len(df_model)} поездок...\n")
            
            df['Расход_л100'] = self.model.predict(df_model[features])
            df['Расход_л'] = df['Расход_л100'] * df['Расстояние_км'] / 100
            
            total_km = df['Расстояние_км'].sum()
            total_fuel = df['Расход_л'].sum()
            avg = df['Расход_л100'].mean()
            
            bus_stats = df.groupby('Госномер').agg({
                'Расход_л': 'sum',
                'Расстояние_км': 'sum',
                'Расход_л100': 'mean'
            }).round(1)
            
            avg_norm = df['Норма_расхода_л'].mean()
            
            # ========== СОХРАНЕНИЕ CSV ФАЙЛОВ ==========
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            
            # Детальный файл поездок
            df_detail = df[[
                'Дата', 'Госномер', 'Район', 'Расстояние_км', 
                'Расход_л', 'Расход_л100', 'Норма_расхода_л'
            ]].copy()
            df_detail.columns = ['Дата', 'Автобус', 'Район', 'Расстояние_км', 
                                 'Расход_л', 'Расход_л100_прогноз', 'Норма_л100км']
            
            detail_filename = f'прогноз_поездки_{contract}_{timestamp}.csv'
            df_detail.to_csv(detail_filename, index=False, encoding='utf-8-sig')
            self._update_results_safe(f"\n💾 Детальный файл: {detail_filename}\n")
            
            # Сводка по автобусам
            bus_stats_reset = bus_stats.reset_index()
            bus_stats_reset.columns = ['Автобус', 'Расход_л_сумма', 'Пробег_км', 'Расход_л100км_средний']
            summary_filename = f'прогноз_сводка_{contract}_{timestamp}.csv'
            bus_stats_reset.to_csv(summary_filename, index=False, encoding='utf-8-sig')
            self._update_results_safe(f"💾 Сводка по автобусам: {summary_filename}\n")
            
            # ========== ВЫВОД РЕЗУЛЬТАТОВ ==========
            self._update_results_safe("\n" + "="*60 + "\n")
            self._update_results_safe("📊 РЕЗУЛЬТАТЫ ПРОГНОЗА\n")
            self._update_results_safe("="*60 + "\n\n")
            self._update_results_safe(f"📅 Период: {start_date.strftime('%d.%m.%Y')} — {end_date.strftime('%d.%m.%Y')}\n")
            self._update_results_safe(f"🚌 Всего поездок: {len(df)}\n")
            self._update_results_safe(f"📏 Общий пробег: {total_km:,.0f} км\n")
            self._update_results_safe(f"⛽ Общий расход: {total_fuel:,.0f} л\n")
            self._update_results_safe(f"📊 Средний расход по модели: {avg:.1f} л/100км\n")
            self._update_results_safe(f"📊 Средняя норма по контракту: {avg_norm:.1f} л/100км\n")
            self._update_results_safe(f"📈 Отклонение от нормы: {avg - avg_norm:+.1f} л/100км\n")
            
            self._update_results_safe(f"\n📊 Статистика по автобусам:\n")
            for bus, row in bus_stats.iterrows():
                self._update_results_safe(f"   {bus}: {row['Расстояние_км']:,.0f} км, {row['Расход_л']:,.0f} л, {row['Расход_л100']:.1f} л/100км\n")
            
            fuel_price = 68
            total_cost = total_fuel * fuel_price
            self._update_results_safe(f"\n💰 Затраты на топливо (при {fuel_price} руб/л):")
            self._update_results_safe(f"   {total_cost:,.0f} руб\n")
            self._update_results_safe("="*60 + "\n")
            
            self.root.after(0, lambda: self.status_var.set("Готов"))
            
        except Exception as e:
            self._update_results_safe(f"❌ Ошибка: {e}\n")
            self._update_results_safe(traceback.format_exc())
            self.root.after(0, lambda: self.status_var.set("Ошибка"))

if __name__ == "__main__":
    root = tk.Tk()
    app = ClassicFuelApp(root)
    root.mainloop()

✅ Загружено 217 автобусов
✅ Загружен fuel_norms_all.csv: 104146 записей
✅ Загружены нормы для 280 автобусов
   А 219 ХВ 198: 30.0 л/100км
   А147АУ778: 10.0 л/100км
   А157АУ778: 10.0 л/100км
   А160АУ778: 10.0 л/100км
   А165АН198: 35.0 л/100км
✅ Модель загружена: best_model.pkl
   Модель ожидает признаки: [np.str_('Distance (km)'), np.str_('Hour'), np.str_('Day of the week'), np.str_('Weekend'), np.str_('Month (sin)'), np.str_('Month (cos)'), np.str_('Bus code'), np.str_('District code'), np.str_('Consumption rate (l/100km)'), np.str_('is winter'), np.str_('is summer'), np.str_('is night'), np.str_('is rush hour')]
✅ Энкодер автобусов загружен
✅ Энкодер районов загружен
